# Deployment-Oriented Operational Evaluation of Unsupervised IDS Models (v0.5.1)

This notebook reframes the work as a **deployment-oriented IDS evaluation** rather than a generic autoencoder exercise.

## Core evaluation principles

1. **Leakage-resistant evaluation protocol**
   - file-disjoint train / validation / test
   - no row-level random leakage
   - sequence integrity preserved for temporal models

2. **Attack-centric evaluation**
   - attack = positive class
   - report precision, recall, F1, ROC-AUC, PR-AUC
   - explicitly report false positive rate on normal traffic

3. **Threshold-sensitive operational analysis**
   - evaluate multiple validation-derived thresholds
   - compare models at several alert budgets / target FPR levels
   - avoid relying on one arbitrary operating point

4. **Ablation evidence**
   - allow comparison between the original dense autoencoder and a more complex hybrid variant
   - test whether added complexity improves recall or merely increases false alarms

5. **Baseline realism**
   - compare dense AE, Isolation Forest, One-Class SVM, and LSTM AE
   - do not assume deep models automatically outperform simpler baselines

## Intended use

Run the notebook end-to-end, then share the exported tables / plots so we can draft the manuscript around the **operational trade-offs** that the results actually support.


## 1. Imports and Configuration


In [ ]:
import os
import gc
import glob
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import tensorflow as tf
from tensorflow.keras import layers, callbacks, regularizers, optimizers
from tensorflow.keras.models import Model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix,
)
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import SGDOneClassSVM
from sklearn.kernel_approximation import Nystroem
from sklearn.covariance import EmpiricalCovariance

warnings.filterwarnings("ignore")

SEED = 67
NORMAL_LABEL = 0
ATTACK_LABEL = 1

MAX_ATTACK_SAMPLES_PER_FILE = 100000
THRESHOLD_PERCENTILES = [90, 95, 97, 98, 99, 99.5]
TARGET_FPR_BUDGETS = [0.01, 0.02, 0.05, 0.10]
REFERENCE_PERCENTILE = 95

RUN_LSTM = True
RUN_HYBRID_ABLATION = True   # Set False if you want a faster first pass
LSTM_SEQ_LEN = 20

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
gc.enable()

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
sns.set_style("whitegrid")

print("Libraries imported successfully")
print(f"Seed: {SEED}")
print(f"Threshold percentiles: {THRESHOLD_PERCENTILES}")
print(f"Target FPR budgets: {TARGET_FPR_BUDGETS}")
print(f"Run LSTM: {RUN_LSTM}")
print(f"Run hybrid ablation: {RUN_HYBRID_ABLATION}")


## 2. Load Baseline Data

The split below is **file-disjoint** and therefore leakage-resistant at the capture-file level.


In [ ]:
DATA_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
all_df = pd.read_parquet(os.path.join(DATA_DIR, 'all.parquet'))

METADATA_COLS = ['pcap_file', 'window_id', 'win_start_epoch']
FEAT_COLS = [c for c in all_df.columns if c not in METADATA_COLS]

TRAIN_FILES = (
    [f'Baseline-{str(i).zfill(3)}.pcapng' for i in range(1, 17)] +
    [f'Baseline-30min-{str(i).zfill(3)}.pcapng' for i in range(1, 7)]
)
VAL_FILES  = [f'Baseline-{str(i).zfill(3)}.pcapng' for i in range(17, 21)]
TEST_FILES = ['Baseline-30min-007.pcapng', 'Baseline-60min-001.pcapng', 'Baseline-60min-002.pcapng']

def load_split(file_list, label):
    subset = all_df[all_df['pcap_file'].isin(file_list)].copy()
    subset = subset.sort_values(['pcap_file', 'window_id'])
    df = subset[FEAT_COLS]
    print(f"\n-- {label} --")
    for f in sorted(file_list):
        n = len(all_df[all_df['pcap_file'] == f])
        print(f"  {f:40s} - {n:,} windows")
    print(f"  -> {label} total: {len(df):,} windows")
    del subset
    gc.collect()
    return df

train_df = load_split(TRAIN_FILES, "TRAIN")
val_df   = load_split(VAL_FILES, "VAL")
test_df  = load_split(TEST_FILES, "TEST")

print(f"\nFeature columns: {len(FEAT_COLS)}")
print(FEAT_COLS[:10], "..." if len(FEAT_COLS) > 10 else "")


In [ ]:
train_raw = np.nan_to_num(train_df.values, nan=0.0, posinf=0.0, neginf=0.0)
val_raw   = np.nan_to_num(val_df.values,   nan=0.0, posinf=0.0, neginf=0.0)
test_raw  = np.nan_to_num(test_df.values,  nan=0.0, posinf=0.0, neginf=0.0)

print(f"Train raw shape: {train_raw.shape}")
print(f"Val raw shape:   {val_raw.shape}")
print(f"Test raw shape:  {test_raw.shape}")


## 3. Scaling Views

We keep two scaling views:

- **MinMaxScaler** for the original dense AE, the classical baselines, and the LSTM AE
- **RobustScaler(5,95)** for the optional optimized hybrid ablation

This makes it possible to compare a simple reference pipeline with a more complex alternative without mixing their preprocessing assumptions.


In [ ]:
scaler_minmax = MinMaxScaler()
train_mm = scaler_minmax.fit_transform(train_raw).astype(np.float32)
val_mm   = scaler_minmax.transform(val_raw).astype(np.float32)
test_mm  = scaler_minmax.transform(test_raw).astype(np.float32)

scaler_robust = RobustScaler(quantile_range=(5, 95))
train_rb = scaler_robust.fit_transform(train_raw).astype(np.float32)
val_rb   = scaler_robust.transform(val_raw).astype(np.float32)
test_rb  = scaler_robust.transform(test_raw).astype(np.float32)

print("MinMax view:")
print(f"  Train: {train_mm.shape}")
print(f"  Val:   {val_mm.shape}")
print(f"  Test:  {test_mm.shape}")

print("\nRobust view:")
print(f"  Train: {train_rb.shape}")
print(f"  Val:   {val_rb.shape}")
print(f"  Test:  {test_rb.shape}")


## 4. Operational Evaluation Helpers


In [ ]:
def reconstruction_mae(model, data, batch_size=2048):
    recon = model.predict(data, batch_size=batch_size, verbose=0)
    err = np.mean(np.abs(recon - data), axis=1)
    return err

def robust_stats(x):
    med = float(np.median(x))
    mad = float(np.median(np.abs(x - med)))
    mad = max(mad, 1e-6)
    return med, mad

def robust_zscore(x, med, mad):
    return 0.6745 * (x - med) / mad

def sample_contiguous_block(data, max_len):
    if len(data) <= max_len:
        return data
    start = random.randint(0, len(data) - max_len)
    return data[start:start + max_len]

def operational_metrics(normal_scores, dos_scores, fdi_scores, threshold):
    attack_scores = np.concatenate([dos_scores, fdi_scores])
    scores = np.concatenate([normal_scores, attack_scores])
    labels = np.concatenate([
        np.full(len(normal_scores), NORMAL_LABEL, dtype=int),
        np.full(len(attack_scores), ATTACK_LABEL, dtype=int),
    ])
    preds = (scores > threshold).astype(int)
    cm = confusion_matrix(labels, preds, labels=[NORMAL_LABEL, ATTACK_LABEL])
    tn, fp, fn, tp = cm.ravel()

    dos_recall = float(np.mean(dos_scores > threshold)) if len(dos_scores) else np.nan
    fdi_recall = float(np.mean(fdi_scores > threshold)) if len(fdi_scores) else np.nan

    metrics = {
        "threshold": float(threshold),
        "accuracy": accuracy_score(labels, preds),
        "attack_precision": precision_score(labels, preds, pos_label=ATTACK_LABEL, zero_division=0),
        "attack_recall": recall_score(labels, preds, pos_label=ATTACK_LABEL, zero_division=0),
        "attack_f1": f1_score(labels, preds, pos_label=ATTACK_LABEL, zero_division=0),
        "roc_auc": roc_auc_score(labels, scores),
        "pr_auc": average_precision_score(labels, scores),
        "fpr": fp / (tn + fp) if (tn + fp) else np.nan,
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "tp": int(tp),
        "normal_windows": int(tn + fp),
        "attack_windows": int(fn + tp),
        "dos_recall": dos_recall,
        "fdi_recall": fdi_recall,
        "dos_missed": int(np.sum(dos_scores <= threshold)),
        "fdi_missed": int(np.sum(fdi_scores <= threshold)),
        "alerts_per_1k_normals": (fp / (tn + fp) * 1000.0) if (tn + fp) else np.nan,
    }
    return metrics

def evaluate_threshold_grid(model_name, val_scores, test_scores, dos_scores, fdi_scores, percentiles=THRESHOLD_PERCENTILES):
    rows = []
    for pct in percentiles:
        thr = np.percentile(val_scores, pct)
        m = operational_metrics(test_scores, dos_scores, fdi_scores, thr)
        m["model"] = model_name
        m["threshold_percentile"] = pct
        rows.append(m)
    return pd.DataFrame(rows)

def evaluate_fpr_budgets(model_name, val_scores, test_scores, dos_scores, fdi_scores, budgets=TARGET_FPR_BUDGETS):
    rows = []
    for budget in budgets:
        pct = 100.0 * (1.0 - budget)
        thr = np.percentile(val_scores, pct)
        m = operational_metrics(test_scores, dos_scores, fdi_scores, thr)
        m["model"] = model_name
        m["target_fpr_budget"] = budget
        m["threshold_percentile"] = pct
        rows.append(m)
    return pd.DataFrame(rows)

def print_reference_metrics(name, metrics):
    print(f"\n{name}")
    print("-" * len(name))
    print(f"Threshold:            {metrics['threshold']:.6f}")
    print(f"Attack precision:     {metrics['attack_precision']*100:.2f}%")
    print(f"Attack recall:        {metrics['attack_recall']*100:.2f}%")
    print(f"Attack F1:            {metrics['attack_f1']*100:.2f}%")
    print(f"ROC-AUC:              {metrics['roc_auc']:.4f}")
    print(f"PR-AUC:               {metrics['pr_auc']:.4f}")
    print(f"False positive rate:  {metrics['fpr']*100:.2f}%")
    print(f"Alerts / 1k normals:  {metrics['alerts_per_1k_normals']:.2f}")
    print(f"DoS recall:           {metrics['dos_recall']*100:.2f}%")
    print(f"FDI recall:           {metrics['fdi_recall']*100:.2f}%")
    print(f"Missed DoS:           {metrics['dos_missed']:,}")
    print(f"Missed FDI:           {metrics['fdi_missed']:,}")

def plot_operating_tradeoff(df, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    x = df["fpr"] * 100
    axes[0].plot(x, df["attack_recall"] * 100, marker="o", linewidth=2)
    axes[0].set_xlabel("Test FPR (%)")
    axes[0].set_ylabel("Attack Recall (%)")
    axes[0].set_title(f"{model_name}: Recall vs FPR")

    axes[1].plot(x, df["attack_precision"] * 100, marker="o", linewidth=2)
    axes[1].set_xlabel("Test FPR (%)")
    axes[1].set_ylabel("Attack Precision (%)")
    axes[1].set_title(f"{model_name}: Precision vs FPR")

    axes[2].plot(x, df["attack_f1"] * 100, marker="o", linewidth=2)
    axes[2].set_xlabel("Test FPR (%)")
    axes[2].set_ylabel("Attack F1 (%)")
    axes[2].set_title(f"{model_name}: F1 vs FPR")

    for ax in axes:
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_score_distribution(normal_scores, dos_scores, fdi_scores, threshold, title, sample_size=30000):
    rng = np.random.default_rng(SEED)
    normal_s = rng.choice(normal_scores, size=min(sample_size, len(normal_scores)), replace=False)
    dos_s = rng.choice(dos_scores, size=min(sample_size, len(dos_scores)), replace=False)
    fdi_s = rng.choice(fdi_scores, size=min(sample_size, len(fdi_scores)), replace=False)
    all_s = np.concatenate([normal_s, dos_s, fdi_s])

    xmin = np.percentile(all_s, 0.5)
    xmax = np.percentile(all_s, 99.0)

    plt.figure(figsize=(14, 5))
    sns.histplot(normal_s, bins=100, color="green", alpha=0.45, label="Normal (test)")
    sns.histplot(dos_s, bins=100, color="red", alpha=0.40, label="DoS")
    sns.histplot(fdi_s, bins=100, color="orange", alpha=0.40, label="FDI")
    plt.axvline(threshold, color="blue", linestyle="--", linewidth=2, label=f"Threshold = {threshold:.4f}")
    plt.xlim(xmin, xmax)
    plt.xlabel("Anomaly score")
    plt.ylabel("Count")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

def collect_family_scores(prefix, scaler, score_fn, max_samples_per_file=MAX_ATTACK_SAMPLES_PER_FILE):
    attack_df = all_df[all_df["pcap_file"].str.startswith(prefix)].copy()
    rows = []
    all_scores = []

    for fname, grp in sorted(attack_df.groupby("pcap_file"), key=lambda x: x[0]):
        raw = np.nan_to_num(grp[FEAT_COLS].values, nan=0.0, posinf=0.0, neginf=0.0)
        raw = sample_contiguous_block(raw, max_samples_per_file)
        scaled = scaler.transform(raw).astype(np.float32)
        scores = np.asarray(score_fn(scaled), dtype=float).reshape(-1)

        rows.append({
            "family": prefix,
            "pcap_file": fname,
            "windows_used": int(len(scores)),
            "mean_score": float(np.mean(scores)),
            "std_score": float(np.std(scores)),
        })
        all_scores.extend(scores.tolist())

        del grp, raw, scaled, scores
        gc.collect()

    return np.asarray(all_scores, dtype=float), pd.DataFrame(rows)

def add_reference_detection_columns(df, threshold):
    out = df.copy()
    out["detected@reference"] = np.nan
    out["reference_threshold"] = threshold
    return out


## 5. Original Dense Autoencoder (Reference Model)

This is the student's original dense reconstruction model, kept as the **reference dense AE**.


In [ ]:
n_input = train_mm.shape[1]

class DenseReferenceAE(Model):
    def __init__(self, n_input):
        super().__init__()
        self.encoder = tf.keras.Sequential([
            layers.Dense(32, activation="relu"),
            layers.Dense(16, activation="relu"),
            layers.Dense(8, activation="relu"),
        ])
        self.decoder = tf.keras.Sequential([
            layers.Dense(16, activation="relu"),
            layers.Dense(32, activation="relu"),
            layers.Dense(n_input, activation="sigmoid"),
        ])

    def call(self, x):
        return self.decoder(self.encoder(x))

dense_ae = DenseReferenceAE(n_input)
dense_ae.compile(optimizer="adam", loss="mae")

print(f"Reference dense AE architecture: {n_input} -> 32 -> 16 -> 8 -> 16 -> 32 -> {n_input}")


In [ ]:
early_stop_dense = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

print("Training reference dense AE...")
history_dense = dense_ae.fit(
    train_mm, train_mm,
    epochs=100,
    batch_size=512,
    validation_data=(val_mm, val_mm),
    callbacks=[early_stop_dense],
    verbose=1,
)
print(f"Training complete, stopped at epoch {len(history_dense.history['loss'])}")


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_dense.history["loss"], label="Training", linewidth=2)
plt.plot(history_dense.history["val_loss"], label="Validation", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss (MAE)")
plt.title("Reference Dense AE Training History")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
dense_val_scores = reconstruction_mae(dense_ae, val_mm)
dense_test_scores = reconstruction_mae(dense_ae, test_mm)
dense_reference_threshold = np.percentile(dense_val_scores, REFERENCE_PERCENTILE)

dense_dos_scores, dense_dos_files = collect_family_scores(
    "DOS",
    scaler_minmax,
    lambda x: reconstruction_mae(dense_ae, x),
)
dense_fdi_scores, dense_fdi_files = collect_family_scores(
    "FDI",
    scaler_minmax,
    lambda x: reconstruction_mae(dense_ae, x),
)

dense_reference_metrics = operational_metrics(
    dense_test_scores,
    dense_dos_scores,
    dense_fdi_scores,
    dense_reference_threshold,
)

print_reference_metrics("Reference Dense AE @ 95th percentile", dense_reference_metrics)
plot_score_distribution(
    dense_test_scores,
    dense_dos_scores,
    dense_fdi_scores,
    dense_reference_threshold,
    "Reference Dense AE: Score Distribution (threshold from validation)",
)


## 6. Operational Analysis for the Reference Dense AE

The goal here is to show how the **same model** behaves under different alert budgets, rather than making claims from one threshold alone.


In [ ]:
dense_threshold_df = evaluate_threshold_grid(
    "Dense AE",
    dense_val_scores,
    dense_test_scores,
    dense_dos_scores,
    dense_fdi_scores,
)

dense_budget_df = evaluate_fpr_budgets(
    "Dense AE",
    dense_val_scores,
    dense_test_scores,
    dense_dos_scores,
    dense_fdi_scores,
)

print("Threshold sensitivity (Dense AE)")
display(dense_threshold_df[[
    "threshold_percentile", "threshold", "attack_precision", "attack_recall", "attack_f1",
    "fpr", "dos_recall", "fdi_recall", "dos_missed", "fdi_missed", "roc_auc", "pr_auc"
]])

print("\nAlert-budget view (Dense AE)")
display(dense_budget_df[[
    "target_fpr_budget", "threshold_percentile", "threshold", "attack_precision", "attack_recall",
    "attack_f1", "fpr", "alerts_per_1k_normals", "dos_recall", "fdi_recall"
]])

plot_operating_tradeoff(dense_threshold_df, "Dense AE")


## 7. Isolation Forest


In [ ]:
print("Training Isolation Forest...")
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=SEED,
    n_jobs=-1,
)
iso_forest.fit(train_mm)
print("Done.")

iso_val_scores = -iso_forest.decision_function(val_mm)
iso_test_scores = -iso_forest.decision_function(test_mm)
iso_dos_scores, iso_dos_files = collect_family_scores(
    "DOS",
    scaler_minmax,
    lambda x: -iso_forest.decision_function(x),
)
iso_fdi_scores, iso_fdi_files = collect_family_scores(
    "FDI",
    scaler_minmax,
    lambda x: -iso_forest.decision_function(x),
)

iso_reference_threshold = np.percentile(iso_val_scores, REFERENCE_PERCENTILE)
iso_reference_metrics = operational_metrics(
    iso_test_scores, iso_dos_scores, iso_fdi_scores, iso_reference_threshold
)
print_reference_metrics("Isolation Forest @ 95th percentile", iso_reference_metrics)

iso_threshold_df = evaluate_threshold_grid(
    "Isolation Forest",
    iso_val_scores, iso_test_scores, iso_dos_scores, iso_fdi_scores
)
iso_budget_df = evaluate_fpr_budgets(
    "Isolation Forest",
    iso_val_scores, iso_test_scores, iso_dos_scores, iso_fdi_scores
)


## 8. One-Class SVM


In [ ]:
gamma_scale = 1.0 / (train_mm.shape[1] * train_mm.var())

print(f"Fitting Nystroem RBF approximation (gamma={gamma_scale:.8f})...")
feature_map = Nystroem(gamma=gamma_scale, n_components=100, random_state=SEED)
train_mapped = feature_map.fit_transform(train_mm)

print("Training SGD One-Class SVM...")
ocsvm_sgd = SGDOneClassSVM(nu=0.05, random_state=SEED)
ocsvm_sgd.fit(train_mapped)
print("Done.")

class WrappedOCSVM:
    def decision_function(self, X):
        return ocsvm_sgd.decision_function(feature_map.transform(X))

ocsvm = WrappedOCSVM()

ocsvm_val_scores = -ocsvm.decision_function(val_mm)
ocsvm_test_scores = -ocsvm.decision_function(test_mm)
ocsvm_dos_scores, ocsvm_dos_files = collect_family_scores(
    "DOS",
    scaler_minmax,
    lambda x: -ocsvm.decision_function(x),
)
ocsvm_fdi_scores, ocsvm_fdi_files = collect_family_scores(
    "FDI",
    scaler_minmax,
    lambda x: -ocsvm.decision_function(x),
)

ocsvm_reference_threshold = np.percentile(ocsvm_val_scores, REFERENCE_PERCENTILE)
ocsvm_reference_metrics = operational_metrics(
    ocsvm_test_scores, ocsvm_dos_scores, ocsvm_fdi_scores, ocsvm_reference_threshold
)
print_reference_metrics("One-Class SVM @ 95th percentile", ocsvm_reference_metrics)

ocsvm_threshold_df = evaluate_threshold_grid(
    "One-Class SVM",
    ocsvm_val_scores, ocsvm_test_scores, ocsvm_dos_scores, ocsvm_fdi_scores
)
ocsvm_budget_df = evaluate_fpr_budgets(
    "One-Class SVM",
    ocsvm_val_scores, ocsvm_test_scores, ocsvm_dos_scores, ocsvm_fdi_scores
)


## 9. LSTM Autoencoder

Sequences are built **within each capture file** to preserve temporal integrity and avoid artificial cross-file sequences.


In [ ]:
if RUN_LSTM:
    def make_sequences_from_array(arr, seq_len):
        n = (len(arr) // seq_len) * seq_len
        if n == 0:
            return np.empty((0, seq_len, arr.shape[1]), dtype=arr.dtype), 0
        dropped = len(arr) - n
        return arr[:n].reshape(-1, seq_len, arr.shape[1]), dropped

    def make_sequences_from_files(file_list, scaler, seq_len):
        seqs = []
        total_windows = 0
        total_used = 0
        total_dropped = 0

        for fname in file_list:
            grp = all_df[all_df["pcap_file"] == fname].sort_values("window_id")
            raw = np.nan_to_num(grp[FEAT_COLS].values, nan=0.0, posinf=0.0, neginf=0.0)
            scaled = scaler.transform(raw).astype(np.float32)

            total_windows += len(scaled)
            file_seqs, dropped = make_sequences_from_array(scaled, LSTM_SEQ_LEN)
            total_used += len(scaled) - dropped
            total_dropped += dropped

            if len(file_seqs) > 0:
                seqs.append(file_seqs)

            del grp, raw, scaled, file_seqs
            gc.collect()

        if not seqs:
            empty = np.empty((0, LSTM_SEQ_LEN, len(FEAT_COLS)), dtype=np.float32)
            return empty, total_windows, total_used, total_dropped

        return np.concatenate(seqs, axis=0), total_windows, total_used, total_dropped

    train_seq, train_windows, train_used, train_dropped = make_sequences_from_files(TRAIN_FILES, scaler_minmax, LSTM_SEQ_LEN)
    val_seq, val_windows, val_used, val_dropped = make_sequences_from_files(VAL_FILES, scaler_minmax, LSTM_SEQ_LEN)
    test_seq, test_windows, test_used, test_dropped = make_sequences_from_files(TEST_FILES, scaler_minmax, LSTM_SEQ_LEN)

    print(f"Train sequences: {train_seq.shape} | used {train_used:,}/{train_windows:,} | dropped {train_dropped:,}")
    print(f"Val sequences:   {val_seq.shape} | used {val_used:,}/{val_windows:,} | dropped {val_dropped:,}")
    print(f"Test sequences:  {test_seq.shape} | used {test_used:,}/{test_windows:,} | dropped {test_dropped:,}")

    inp = tf.keras.Input(shape=(LSTM_SEQ_LEN, n_input))
    x = layers.LSTM(32, return_sequences=True)(inp)
    enc = layers.LSTM(16)(x)
    x = layers.RepeatVector(LSTM_SEQ_LEN)(enc)
    x = layers.LSTM(16, return_sequences=True)(x)
    x = layers.LSTM(32, return_sequences=True)(x)
    out = layers.TimeDistributed(layers.Dense(n_input, activation="sigmoid"))(x)

    lstm_ae = Model(inp, out, name="lstm_autoencoder")
    lstm_ae.compile(optimizer="adam", loss="mae")
    lstm_ae.summary()

    early_stop_lstm = callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    )

    print("Training LSTM autoencoder...")
    history_lstm = lstm_ae.fit(
        train_seq, train_seq,
        epochs=100,
        batch_size=512,
        validation_data=(val_seq, val_seq),
        callbacks=[early_stop_lstm],
        verbose=1,
    )
    print(f"Training complete, stopped at epoch {len(history_lstm.history['loss'])}")

    plt.figure(figsize=(10, 5))
    plt.plot(history_lstm.history["loss"], label="Training", linewidth=2)
    plt.plot(history_lstm.history["val_loss"], label="Validation", linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss (MAE)")
    plt.title("LSTM AE Training History")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    lstm_val_recon = lstm_ae.predict(val_seq, verbose=0)
    lstm_val_scores = np.mean(np.abs(lstm_val_recon - val_seq), axis=-1).flatten()

    lstm_test_recon = lstm_ae.predict(test_seq, verbose=0)
    lstm_test_scores = np.mean(np.abs(lstm_test_recon - test_seq), axis=-1).flatten()

    def collect_lstm_family_scores(prefix, scaler, seq_len=LSTM_SEQ_LEN, max_samples_per_file=MAX_ATTACK_SAMPLES_PER_FILE):
        attack_df = all_df[all_df["pcap_file"].str.startswith(prefix)].copy()
        rows = []
        all_scores = []

        for fname, grp in sorted(attack_df.groupby("pcap_file"), key=lambda x: x[0]):
            raw = np.nan_to_num(grp[FEAT_COLS].values, nan=0.0, posinf=0.0, neginf=0.0)
            raw = sample_contiguous_block(raw, max_samples_per_file)
            scaled = scaler.transform(raw).astype(np.float32)
            seqs, dropped = make_sequences_from_array(scaled, seq_len)
            if len(seqs) == 0:
                continue

            recon = lstm_ae.predict(seqs, verbose=0)
            scores = np.mean(np.abs(recon - seqs), axis=-1).flatten()

            rows.append({
                "family": prefix,
                "pcap_file": fname,
                "windows_used": int(len(scores)),
                "dropped_windows": int(dropped),
                "mean_score": float(np.mean(scores)),
                "std_score": float(np.std(scores)),
            })
            all_scores.extend(scores.tolist())

            del grp, raw, scaled, seqs, recon, scores
            gc.collect()

        return np.asarray(all_scores, dtype=float), pd.DataFrame(rows)

    lstm_dos_scores, lstm_dos_files = collect_lstm_family_scores("DOS", scaler_minmax)
    lstm_fdi_scores, lstm_fdi_files = collect_lstm_family_scores("FDI", scaler_minmax)

    lstm_reference_threshold = np.percentile(lstm_val_scores, REFERENCE_PERCENTILE)
    lstm_reference_metrics = operational_metrics(
        lstm_test_scores, lstm_dos_scores, lstm_fdi_scores, lstm_reference_threshold
    )
    print_reference_metrics("LSTM AE @ 95th percentile", lstm_reference_metrics)

    lstm_threshold_df = evaluate_threshold_grid(
        "LSTM AE",
        lstm_val_scores, lstm_test_scores, lstm_dos_scores, lstm_fdi_scores
    )
    lstm_budget_df = evaluate_fpr_budgets(
        "LSTM AE",
        lstm_val_scores, lstm_test_scores, lstm_dos_scores, lstm_fdi_scores
    )

    del lstm_val_recon, lstm_test_recon, train_seq, val_seq, test_seq
    gc.collect()
else:
    print("RUN_LSTM is False. Skipping LSTM section.")


## 10. Optional Hybrid Ablation

This section is intentionally framed as an **ablation**, not the default final model.

The purpose is to test a more complex variant:
- RobustScaler(5,95)
- denoising + sparse undercomplete AE
- hybrid score = reconstruction branch + latent Mahalanobis branch

If this raises false positives without materially improving recall, that is a useful **negative operational finding**.


In [ ]:
if RUN_HYBRID_ABLATION:
    LATENT_DIM = 8
    NOISE_STD = 0.02
    DROPOUT_RATE = 0.10
    L2_REG = 1e-5
    L1_ACTIVITY = 1e-5

    class HybridAblationAE(Model):
        def __init__(self, n_input, latent_dim=LATENT_DIM):
            super().__init__()
            self.noise = layers.GaussianNoise(NOISE_STD)
            self.encoder = tf.keras.Sequential([
                layers.Dense(64, kernel_regularizer=regularizers.l2(L2_REG)),
                layers.BatchNormalization(),
                layers.Activation("relu"),
                layers.Dropout(DROPOUT_RATE),
                layers.Dense(32, kernel_regularizer=regularizers.l2(L2_REG)),
                layers.BatchNormalization(),
                layers.Activation("relu"),
                layers.Dense(
                    latent_dim,
                    activation=None,
                    activity_regularizer=regularizers.l1(L1_ACTIVITY),
                    name="latent",
                ),
            ])
            self.decoder = tf.keras.Sequential([
                layers.Dense(32, kernel_regularizer=regularizers.l2(L2_REG)),
                layers.BatchNormalization(),
                layers.Activation("relu"),
                layers.Dense(64, kernel_regularizer=regularizers.l2(L2_REG)),
                layers.BatchNormalization(),
                layers.Activation("relu"),
                layers.Dense(n_input, activation="linear"),
            ])

        def call(self, x, training=False):
            x_noisy = self.noise(x, training=training)
            z = self.encoder(x_noisy, training=training)
            return self.decoder(z, training=training)

        def encode(self, x, training=False):
            return self.encoder(x, training=training)

    hybrid_ae = HybridAblationAE(n_input)
    hybrid_ae.compile(
        optimizer=optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-5),
        loss=tf.keras.losses.Huber(delta=1.0),
    )

    early_stop_hybrid = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        min_delta=1e-4,
    )
    reduce_lr_hybrid = callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1,
    )

    print("Training hybrid ablation AE...")
    history_hybrid = hybrid_ae.fit(
        train_rb, train_rb,
        epochs=150,
        batch_size=1024,
        validation_data=(val_rb, val_rb),
        callbacks=[early_stop_hybrid, reduce_lr_hybrid],
        verbose=1,
        shuffle=True,
    )
    print(f"Training complete, stopped at epoch {len(history_hybrid.history['loss'])}")

    plt.figure(figsize=(10, 5))
    plt.plot(history_hybrid.history["loss"], label="Training", linewidth=2)
    plt.plot(history_hybrid.history["val_loss"], label="Validation", linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Hybrid Ablation AE Training History")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    hybrid_train_recon = reconstruction_mae(hybrid_ae, train_rb)
    hybrid_val_recon = reconstruction_mae(hybrid_ae, val_rb)
    hybrid_test_recon = reconstruction_mae(hybrid_ae, test_rb)

    hybrid_train_latent = hybrid_ae.encode(train_rb, training=False).numpy()
    hybrid_val_latent = hybrid_ae.encode(val_rb, training=False).numpy()
    hybrid_test_latent = hybrid_ae.encode(test_rb, training=False).numpy()

    latent_model = EmpiricalCovariance()
    latent_model.fit(hybrid_train_latent)

    hybrid_train_md = latent_model.mahalanobis(hybrid_train_latent)
    hybrid_val_md = latent_model.mahalanobis(hybrid_val_latent)
    hybrid_test_md = latent_model.mahalanobis(hybrid_test_latent)

    recon_med, recon_mad = robust_stats(hybrid_train_recon)
    md_med, md_mad = robust_stats(hybrid_train_md)

    hybrid_val_scores = robust_zscore(hybrid_val_recon, recon_med, recon_mad) + robust_zscore(hybrid_val_md, md_med, md_mad)
    hybrid_test_scores = robust_zscore(hybrid_test_recon, recon_med, recon_mad) + robust_zscore(hybrid_test_md, md_med, md_mad)

    def score_hybrid_batch(data_rb):
        recon_err = reconstruction_mae(hybrid_ae, data_rb)
        latent = hybrid_ae.encode(data_rb, training=False).numpy()
        latent_md = latent_model.mahalanobis(latent)
        scores = robust_zscore(recon_err, recon_med, recon_mad) + robust_zscore(latent_md, md_med, md_mad)
        return scores

    hybrid_dos_scores, hybrid_dos_files = collect_family_scores(
        "DOS",
        scaler_robust,
        score_hybrid_batch,
    )
    hybrid_fdi_scores, hybrid_fdi_files = collect_family_scores(
        "FDI",
        scaler_robust,
        score_hybrid_batch,
    )

    hybrid_reference_threshold = np.percentile(hybrid_val_scores, REFERENCE_PERCENTILE)
    hybrid_reference_metrics = operational_metrics(
        hybrid_test_scores, hybrid_dos_scores, hybrid_fdi_scores, hybrid_reference_threshold
    )
    print_reference_metrics("Hybrid Ablation AE @ 95th percentile", hybrid_reference_metrics)

    hybrid_threshold_df = evaluate_threshold_grid(
        "Hybrid AE",
        hybrid_val_scores, hybrid_test_scores, hybrid_dos_scores, hybrid_fdi_scores
    )
    hybrid_budget_df = evaluate_fpr_budgets(
        "Hybrid AE",
        hybrid_val_scores, hybrid_test_scores, hybrid_dos_scores, hybrid_fdi_scores
    )
else:
    print("RUN_HYBRID_ABLATION is False. Skipping hybrid ablation section.")


## 11. Deployment-Oriented Model Comparison

The tables below are organized around:
- **reference threshold** from the validation 95th percentile
- **threshold sensitivity**
- **alert-budget / FPR-oriented comparison**
- **per-family recall**


In [ ]:
model_results = []

def append_reference_row(model_name, ref_metrics):
    row = {
        "Model": model_name,
        "Threshold@ref": ref_metrics["threshold"],
        "Attack Precision (%)": ref_metrics["attack_precision"] * 100,
        "Attack Recall (%)": ref_metrics["attack_recall"] * 100,
        "Attack F1 (%)": ref_metrics["attack_f1"] * 100,
        "Test FPR (%)": ref_metrics["fpr"] * 100,
        "Alerts per 1k normals": ref_metrics["alerts_per_1k_normals"],
        "DoS Recall (%)": ref_metrics["dos_recall"] * 100,
        "FDI Recall (%)": ref_metrics["fdi_recall"] * 100,
        "Missed DoS": ref_metrics["dos_missed"],
        "Missed FDI": ref_metrics["fdi_missed"],
        "ROC-AUC": ref_metrics["roc_auc"],
        "PR-AUC": ref_metrics["pr_auc"],
    }
    model_results.append(row)

append_reference_row("Isolation Forest", iso_reference_metrics)
append_reference_row("One-Class SVM", ocsvm_reference_metrics)
append_reference_row("Dense AE", dense_reference_metrics)

if RUN_LSTM:
    append_reference_row("LSTM AE", lstm_reference_metrics)

if RUN_HYBRID_ABLATION:
    append_reference_row("Hybrid AE", hybrid_reference_metrics)

reference_df = pd.DataFrame(model_results).sort_values(by=["Attack F1 (%)", "Attack Recall (%)"], ascending=False)
print("Reference-threshold comparison (95th percentile on validation)")
display(reference_df)


In [ ]:
budget_frames = [iso_budget_df, ocsvm_budget_df, dense_budget_df]
if RUN_LSTM:
    budget_frames.append(lstm_budget_df)
if RUN_HYBRID_ABLATION:
    budget_frames.append(hybrid_budget_df)

all_budget_df = pd.concat(budget_frames, ignore_index=True)
all_budget_df["target_fpr_budget_pct"] = all_budget_df["target_fpr_budget"] * 100

print("Comparison at multiple alert budgets")
display(
    all_budget_df[[
        "model", "target_fpr_budget_pct", "threshold", "attack_precision", "attack_recall",
        "attack_f1", "fpr", "alerts_per_1k_normals", "dos_recall", "fdi_recall", "roc_auc", "pr_auc"
    ]].sort_values(["target_fpr_budget_pct", "attack_f1"], ascending=[True, False])
)


In [ ]:
threshold_frames = [iso_threshold_df, ocsvm_threshold_df, dense_threshold_df]
if RUN_LSTM:
    threshold_frames.append(lstm_threshold_df)
if RUN_HYBRID_ABLATION:
    threshold_frames.append(hybrid_threshold_df)

all_threshold_df = pd.concat(threshold_frames, ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for model_name, grp in all_threshold_df.groupby("model"):
    axes[0].plot(grp["fpr"] * 100, grp["attack_recall"] * 100, marker="o", linewidth=2, label=model_name)
    axes[1].plot(grp["fpr"] * 100, grp["attack_precision"] * 100, marker="o", linewidth=2, label=model_name)
    axes[2].plot(grp["fpr"] * 100, grp["attack_f1"] * 100, marker="o", linewidth=2, label=model_name)

axes[0].set_title("Recall vs Test FPR")
axes[1].set_title("Precision vs Test FPR")
axes[2].set_title("F1 vs Test FPR")

for ax in axes:
    ax.set_xlabel("Test FPR (%)")
    ax.grid(alpha=0.3)
    ax.legend()

axes[0].set_ylabel("Attack Recall (%)")
axes[1].set_ylabel("Attack Precision (%)")
axes[2].set_ylabel("Attack F1 (%)")

plt.tight_layout()
plt.show()


## 12. Ablation-Focused Summary

If the hybrid ablation produces **little recall gain but much higher FPR**, that is a valid and useful result for a deployment-oriented paper.


In [ ]:
if RUN_HYBRID_ABLATION:
    ablation_rows = []

    dense_ref = dense_reference_metrics
    hybrid_ref = hybrid_reference_metrics

    ablation_rows.append({
        "Variant": "Dense AE (reference)",
        "Attack Precision (%)": dense_ref["attack_precision"] * 100,
        "Attack Recall (%)": dense_ref["attack_recall"] * 100,
        "Attack F1 (%)": dense_ref["attack_f1"] * 100,
        "Test FPR (%)": dense_ref["fpr"] * 100,
        "DoS Recall (%)": dense_ref["dos_recall"] * 100,
        "FDI Recall (%)": dense_ref["fdi_recall"] * 100,
        "ROC-AUC": dense_ref["roc_auc"],
        "PR-AUC": dense_ref["pr_auc"],
    })
    ablation_rows.append({
        "Variant": "Hybrid AE (ablation)",
        "Attack Precision (%)": hybrid_ref["attack_precision"] * 100,
        "Attack Recall (%)": hybrid_ref["attack_recall"] * 100,
        "Attack F1 (%)": hybrid_ref["attack_f1"] * 100,
        "Test FPR (%)": hybrid_ref["fpr"] * 100,
        "DoS Recall (%)": hybrid_ref["dos_recall"] * 100,
        "FDI Recall (%)": hybrid_ref["fdi_recall"] * 100,
        "ROC-AUC": hybrid_ref["roc_auc"],
        "PR-AUC": hybrid_ref["pr_auc"],
    })

    ablation_df = pd.DataFrame(ablation_rows)
    print("Dense AE vs Hybrid AE ablation at the reference threshold")
    display(ablation_df)
else:
    print("Hybrid ablation was skipped.")


## 13. Export Helper for Sharing Results

This cell prints the recommended outputs to share and optionally exports them as CSV files.

In [ ]:
EXPORT_RESULTS = False
EXPORT_DIR = os.path.join(DATA_DIR, "operational_eval_exports")
os.makedirs(EXPORT_DIR, exist_ok=True)

export_candidates = [
    ("reference_df", "reference-threshold comparison"),
    ("all_budget_df", "alert-budget comparison across models"),
    ("all_threshold_df", "threshold sensitivity across models"),
    ("dense_threshold_df", "detailed dense AE operational curve"),
    ("iso_threshold_df", "Isolation Forest operational curve"),
    ("ocsvm_threshold_df", "One-Class SVM operational curve"),
]

if RUN_LSTM:
    export_candidates.append(("lstm_threshold_df", "LSTM AE operational curve"))

if RUN_HYBRID_ABLATION:
    export_candidates.append(("hybrid_threshold_df", "hybrid ablation operational curve"))
    export_candidates.append(("ablation_df", "dense AE vs hybrid AE summary"))

print("Recommended outputs to export and share:")
for idx, (name, desc) in enumerate(export_candidates, start=1):
    status = "available" if name in globals() else "not available"
    print(f"{idx}. {name:<24} -> {desc} [{status}]")

print("\nUse these exported tables and plots for the manuscript narrative.")

if EXPORT_RESULTS:
    exported = []
    for name, _ in export_candidates:
        if name in globals():
            out_path = os.path.join(EXPORT_DIR, f"{name}.csv")
            globals()[name].to_csv(out_path, index=False)
            exported.append(out_path)

    if exported:
        print(f"\nExported {len(exported)} CSV files to: {EXPORT_DIR}")
        for p in exported:
            print(" -", p)
    else:
        print("\nNo tables were exported because none of the expected DataFrames were available.")
else:
    print("\nSet EXPORT_RESULTS = True and re-run this cell to write CSV files.")